# Scaling Up GPU Compute for Large Datasets Using PyTorch DDP

In the previous tutorial, we learned how to scale up our CPU-based computations using Dask. In this tutorial we'll explore how to scale up GPU compute across multiple GPUs using `DistributedDataParallel`, or DDP. Similarly to Dask, DDP manages the distributed scheduling of your model setup and synchronizes gradient descent across multiple GPUs and machines.

We'll start with the same data that we used before.

In [1]:
import numpy as np

In [2]:
n_images = int(1e4)
height = 64
width = 64

images = np.random.random((n_images, height, width)).astype(np.float32)

images.shape

(10000, 64, 64)

When working with datasets at this scale on CPUs, we saw that it was useful to divide the data into smaller “chunks” and process those chunks in parallel. When running GPU-enabled compute, we employ a similar subsetting approach, but data subsets are typically called batches, and in each new epoch, batches are shuffled. In contrast, dask chunks are typically deterministic and remain static throughout the compute pipeline. During a training epoch, each batch is moved onto the GPU and processed independently, much like a dask worker processes a chunk of an array.

One key difference between chunks and batches is in what limits their size. On CPUs, chunk size is often chosen based on balancing parallelism with the overhead required to send tasks to each worker and memory use. On GPUs, batch size is constrained much more directly by memory: each batch must fit alongside the model and any intermediate activations required during computation.

So while the terminology changes from chunks to batches as we transition from multi-CPU to multi-GPU compute, the underlying idea remains the same. In both, we subset a large dataset into manageable subsets, and process those subsets in parallel.

---

<br>

## Loading Data onto a Single GPU Using PyTorch

[PyTorch](https://pytorch.org/) is an open-source deep learning framework that allows you to build your own AI architectures in Python. It provides two core capabilities: an n-dim tensor library with GPU acceleration (similar to a GPU-enabled `numpy`) and an automatic differentiation engine for computing gradients during model training.

The central workflow in PyTorch is a training loop. For each batch of data, you run a forward pass to produce predictions, compute a loss, run a backward pass to compute gradients, and then update your model weights using an optimizer.

In [3]:
import torch
from torch.utils.data import Dataset

PyTorch uses tensors to load data from the CPUs onto GPUs, and this transforms the data into a type that is compatible with GPU compute. We start by defining a custom dataset following the [PyTorch docs](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html). This turns our `numpy` array into an indexable stream of samples. Similar to dask, this allows the dataset to be built lazily. However in contrast to dask this builds tensors lazily from data that is already in CPU memory, whereas dask doesn't pull data in CPU memory until it's needed. Instead, PyTorch doesn't load data into tensors and onto the GPU until it's needed there.

In [4]:
class ImageDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]

Once we have our `Dataset` function defined, we can wrap our `numpy` array into a Pytorch dataset.

In [5]:
dataset = ImageDataset(images)

Now we're going to define a `DataLoader` that loads the data in batches.

In [6]:
from torch.utils.data import DataLoader

In [7]:
loader = DataLoader(dataset, batch_size=128, shuffle=True)

In dask, we were using chunk sizes of 1000 images at a time. Here we are loading batches that have 128 images. These are conceptually similar, but typically in GPU compute we are much more limited in terms of how many images can be analyzed at one time.

Because this tutorial is focused on multi-GPU distribution, we'll deploy a minimal model here. In typical research workflows, you'll likely need a much more complex model which will take up more room on the GPU. Make sure to tailor the batch size you pick to accomodate for the size of the model + activations and the onboard memory available to you in just one of your GPUs.

In [8]:
import torch.nn as nn

In [9]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64 * 64, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

Now we'll run this model and data through a single GPU, training batch by batch. We first detect which device is available. If a CUDA enabled GPU is present, we use that, and otherwise the code defaults to any CPUs that are available. We then move both the model and each batch, one by one, to the device. During this tutorial, we'll be using Sherlock's GPUs as the device, and these are CUDA enabled.

In [10]:
import torch.optim as optim

In [11]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model = model.to(device)

optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

for batch in loader:
    x = batch.to(device)
    y = torch.zeros(x.size(0), dtype=torch.long, device=device)  # dummy labels

    logits = model(x)
    loss = loss_fn(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

Training on: cuda:0


Now we've trained the model on a single GPU, let's scale up to multiple GPUs.

## Using DistributedDataParallel to Distribute to Multiple GPUs

PyTorch's `DistributedDataParallel` (DDP) allows you to train across multiple GPUs by distributing batches to all available GPUs and combining gradient descent computations. At a high level, it distributes the compute as follows:
1. One process per GPU: Each GPU has a full copy of the model. 
2. Data is distributed across process: Each process receives non-overlapping batches of the dataset.
3. Gradients are synchronized: after each backward pass, DDP averages gradients across all processes, keeping every model replica on each GPU in sync.

DDP differs from single-GPU training in a fundamental way. Instead of one process sequentially working through all batches, DDP launches multiple processes, each handling a different subset of the data at the same time. After each forward and backward pass, these processes communicate to average their gradients, keeping model parameters in sync. To make this coordination possible, all processes must be organized into a process group, which enables them to exchange gradients and updates efficiently during training.

These process groups have a few key parameters:
- `backend`: `"nccl"` is the recommended choice for communication between GPUs
- `world_size`: total number of processes (typically nGPUs)
- `rank`: a unique integer ID for each individual process, that extends from 0 to `world_size-1`

In [12]:
import os
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
import torch.multiprocessing as mp

We're going to move our model to train across multiple GPUs. This still means that we'll be measuring gradient descent through the same model across multiple GPUs, so we will need to load the model onto each GPU and distribute batches across the GPUs. Similarly to before, we need to pick batch sizes that fit onto our GPU alongside the model and any required activations.

We need to explicitly launch multiple processes.

In [13]:
def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'

    # nccl is recommended for multi-GPU training
    backend = "nccl" if torch.cuda.is_available() else "gloo"
    dist.init_process_group(backend, rank=rank, world_size=world_size)

def cleanup():
    '''Tear down the process group after training completes to reset GPU'''
    dist.destroy_process_group()

Next, we define the function that each process will run. This function will move the model to `cuda:rank`, wrap it in `DDP`, and create a `DistributedSampler` so each rank gets a unique set of batches. Then it will run the training loop and cleanup. Notice how similar this loop is to all of the components in the single GPU version. This is the elegance of `DDP`, the function does the multi-GPU distribution under the hood.

In [14]:
def train(rank, world_size, dataset):
    '''
    Args:
       rank (int): rank for this specific process
       world_size (int): n processes
       dataset (Dataset): the full dataset shared across all processes
    '''
    setup(rank, world_size)

    # Define device as the specific GPU corresponding to the rank
    device = torch.device(f"cuda:{rank}" if torch.cuda.is_available() else "cpu")

    # Send the model to that rank's device
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64 * 64, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    ).to(device)

    # DDP wraps the model and reduces gradients after each backward pass
    ddp_model = DDP(model, device_ids=[rank] if torch.cuda.is_available() else None)

    # Distributed sampler assigns unique batches to each specific GPU rank
    sampler = DistributedSampler(dataset, num_replicas=world_size, 
                                 rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=128, sampler=sampler)

    optimizer = optim.Adam(ddp_model.parameters())
    loss_fn = nn.CrossEntropyLoss()

    n_epochs = 3
    for epoch in range(n_epochs):
        sampler.set_epoch(epoch) #ensures reshuffling at the start of the epoch

        total_loss = 0.0
        for batch in loader:
            x = batch.to(device)
            y = torch.zeros(x.size(0), dtype=torch.long, device=device)
        
            logits = ddp_model(x)
            loss = loss_fn(logits, y)
        
            optimizer.zero_grad()
            loss.backward() #DDP will automatically reduce these across all GPUs
            optimizer.step()

            total_loss += loss.item()

        # Only print on rank 0 to avoid errors
        if rank == 0:
            print(f"Epoch {epoch + 1}/{n_epochs} | Loss: {total_loss / len(loader):.4f}")
            
    cleanup()

Now that we've defined a trainer that distributes our training using `DDP` in tandem with `DistributedSampler` we can run this. We'll create a workflow that atuomatically detects `world_size` so you can size up how many GPUs you use dynamically based on queue availability.

In [15]:
world_size = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f'Launching {world_size} process/es across {world_size} device/s')

Launching 2 process/es across 2 device/s


If we were working from an interactive shell, we'd use multiprocess to spawn multiple GPU processes, but `mp.spawn()` doesn't interact well with a Jupyter notebook.

```python
mp.spawn(
    train,
    args=(world_size, dataset),
    nprocs=world_size,
    join=True
)
```

Since Jupyter already runs in a managed process, we'll instead save our code to a Python file, `train_ddp.py`. Now we can use the command line launcer `torchrun`, if we use some Jupyter magic to treat the next cell like a bash shell.

In order to run the next cell, we're going to restart our kernel to clear out anything that is currently stored on our GPUs from the notebook development that we did above.

In [1]:
%%bash
python -m torch.distributed.run \
  --nproc_per_node=2 \
  train_ddp.py

[W512 21:27:03.908830969 socket.cpp:752] [c10d] The client socket cannot be initialized to connect to [localhost]:29500 (errno: 97 - Address family not supported by protocol).
[W512 21:27:05.248995729 socket.cpp:752] [c10d] The client socket cannot be initialized to connect to [localhost]:29500 (errno: 97 - Address family not supported by protocol).
[W512 21:27:05.249833417 socket.cpp:752] [c10d] The client socket cannot be initialized to connect to [localhost]:29500 (errno: 97 - Address family not supported by protocol).


Epoch 1/3 | Loss: 0.0570
Epoch 2/3 | Loss: 0.0000
Epoch 3/3 | Loss: 0.0000


A quick note! Because we're working from a container, we can just call
```bash
python -m torch.distributed.run
```
in the cell above this. If you're working directly from Sherlock or a `venv`, it can be useful to name which Python you're using directly. Here's how to find that from Python:

In [16]:
import sys

In [17]:
print(sys.executable)

/opt/conda/envs/torchenv/bin/python


With this information, we would rewrite our distributed run above in the shell as:
```bash
/opt/conda/envs/torchenv/bin/python -m torch.distributed.run \
  --nproc_per_node=2 \
  train_ddp.py
```